#### Import Libraries, archives and Connections

In [ ]:
#Import SQL libraries
from sqlalchemy import create_engine, text, Table, Column, Integer, MetaData, delete, Date
from sqlalchemy.sql import tuple_

#import access.ipynb. This archive needs to be created manually and consider variables to do queries and connect to other apps.
%run ../../access.ipynb

#import 01_disti_sales_format.ipynb format. This will get df_sales_sanitized and df_stock_sanitized
%run "01_disti_sales_format.ipynb"

#Connect to PostgreSQL database. Variables used are imported on access.ipynb
engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}')

#### Insert new [part_number]

Step 1: Look up for sku/resellers in sales and stock dataframes

In [ ]:
#Concatenate sales and stock to get all skus sent by distis
df_sales_stock = pd.concat([df_sales_sanitized, df_stock_sanitized], ignore_index= False)

#Create dataframe to analize sku and part_numbers
df_sku = df_sales_stock[['country_id', 'importer_id', 'sku', 'part_number']]
df_sku = df_sku.drop_duplicates(subset=['sku', 'importer_id']) #dropping duplicates by sku. partnumbers are unique values
df_sku['part_number'] = df_sku['part_number'].str.replace('-REF', '', regex=False) #Delete refurbishes part_number indicator.

Step 2: Import from DB: product_list, business_units

In [ ]:
#Get existing part numbers (pn) from database
pn_query = "SELECT id AS product_id, part_number FROM product_list WHERE brand_id = 6"
df_pn_sql = pd.read_sql(pn_query, engine)

#Get existing BUs from business_unit database
bu_query = "SELECT id AS business_unit_id, bu_initials FROM business_unit"
df_bu_sql = pd.read_sql(bu_query, engine)

Step 3: check if part_numbers are in database tables

In [ ]:
#Identifying part_number that are not found in product_list table
new_pn = df_sku[~df_sku['part_number'].isin(df_pn_sql['part_number'])]
print(f'part_number nuevos: {new_pn.shape}')

#Define BU of new part_numbers in case of 90XX format
#Convert initials from business_unit table into list
bu_initials_sql = df_bu_sql['bu_initials'].tolist()

#Create bu column from third and fourth value from part_number. 
# If value doesn't start with '90' and code not in bu_initials is None
new_pn['bu_initials'] = new_pn['part_number'].apply(lambda x: x[2:4] if x.startswith('90') and x[2:4] in bu_initials_sql else None)
new_pn = pd.merge(new_pn, df_bu_sql, on = 'bu_initials', how = 'left')

#Get only the new part_number of the skus
new_pn = new_pn[['part_number', 'business_unit_id']]
new_pn.drop_duplicates(subset=['part_number'], inplace= True)

#Assign brand to part_numbers. ASUS brand_id is 6
new_pn['brand_id'] = 6

new_pn.to_csv(r'../output/02_new_part_number.csv', index=False)

#Create a copy of new_dataframes df to preserve last changes.
new_pn_to_upload = new_pn.copy()

Step 4: insert new [part_number] values

In [ ]:
if not new_pn_to_upload.empty:
    new_pn_to_upload.to_sql('product_list', engine, if_exists='append', index=False)
    print("Data inserted to 'product_list'.")
else:
    print("No data to upload. The DataFrame is empty.")

#### Insert new sku values to sku_disti table in SQL

Step 1: Import from DB: sku_disti and updated part_number tables

In [ ]:
# SQL query from sku_disti table with sku_id to merge them into dataframe
sku_disti_query = f"""SELECT sku_disti.id AS sku_id, sku, business_unit.bu_initials, importer_id
                    FROM sku_disti 
                    LEFT JOIN importer ON sku_disti.importer_id = importer.id
                    LEFT JOIN product_list ON sku_disti.product_id = product_list.id
                    LEFT JOIN business_unit on product_list.business_unit_id = business_unit.id
                    WHERE country_id = ({country_id_credential})"""

df_sku_disti_sql = pd.read_sql(sku_disti_query, engine)

#Get pn_query updated with the newly added part_numbers
new_df_pn_sql = pd.read_sql(pn_query, engine)

Step 2: check for sku and reseller_name reported by distis in database tables

In [ ]:
#Identifying skus that are not found in database
#We need to check if the combination of [sku, importer_id] is not present in df_sku_disti_sql
new_sku = df_sku.merge(df_sku_disti_sql[['sku', 'importer_id']], on=['sku', 'importer_id'], how='left', indicator=True)
new_sku = new_sku[new_sku['_merge'] == 'left_only'].drop(columns=['_merge'])

print(f'sku nuevos: {new_sku.shape}')

Step 3: formatting new tables into database format

In [ ]:
#Format new_sku like sku_disti table. It should have 3 columns including id
new_sku = pd.merge(new_sku, new_df_pn_sql, on = 'part_number', how = 'left')

#Keep only columns that will be added to sku_disti table
new_sku = new_sku[['sku', 'importer_id', 'product_id']]

new_sku.to_csv(r'../output/02_new_sku.csv', index=False)

#Create a copy of new_dataframes df to preserve last changes.
new_sku_to_upload = new_sku.copy()

Step 4: Replace missing values if any

In [ ]:
#If not found assign value -1
#new_sku_to_upload['product_id'] = new_sku_to_upload['product_id'].fillna().astype('int32')

#Define which columns are going to be checked for sku and reseller dataframes
columns_to_check_sku = ['product_id', 'importer_id']
mask_sku = (new_sku_to_upload[columns_to_check_sku] == -1).any(axis=1)
sku_to_check = new_sku_to_upload[mask_sku]

sku_to_check.to_csv(r'../output/02_sku_missing_ids.csv', index=False)

Step 5: upload new sku into sku_distis table

In [ ]:
if not new_sku_to_upload.empty:
    new_sku_to_upload.to_sql('sku_disti', engine, if_exists='append', index=False)
    print("Data inserted to 'sku_disti'.")
else:
    print("No data to upload. The DataFrame is empty.")

#### Insert new [reseller_names] into reseller_list table in SQL

In [ ]:
#----------------------------------------------------------------------------------------------
# Step 1 ) Prepare DF we are going to use
#----------------------------------------------------------------------------------------------

# Generate df with only columns that are going to be used to upload info to reseller_list table.
reseller_list_columns = ['reseller_name', 'country_segment', 'asus_segment', 'country_id', 'reseller_code', 'importer_id']
reseller_list_df = reseller_formatted[reseller_list_columns]

# Drop Intcomex columns because they don't provide resellers information.
reseller_list_df = reseller_list_df[reseller_list_df['importer_id'] != 5]

#----------------------------------------------------------------------------------------------
# Step 2 ) Filter out rows where [reseller_code] is already in our database
#----------------------------------------------------------------------------------------------

# SQL query to get reseller_code from database
reseller_code_query = f"""SELECT reseller_code
                        FROM reseller_codes
                        LEFT JOIN importer ON reseller_codes.importer_id = importer.id
                        WHERE importer.country_id = 1"""

# Generate df from reseller_query
df_reseller_code_sql = pd.read_sql(reseller_code_query, engine)

# Filter reseller_list_df to show only rows with new [reseller_code] values
# This is because new [reseller_code] values are from new resellers.
reseller_list_df = reseller_list_df[~reseller_list_df['reseller_code'].isin(df_reseller_code_sql['reseller_code'])]

#----------------------------------------------------------------------------------------------
# Step 3 ) Filter out rows where [reseller_name] is already in our database
#----------------------------------------------------------------------------------------------

# SQL query to get reseller_name from database
reseller_name_query = f"""SELECT reseller_name FROM reseller_list
                        WHERE country_id = 1
                        AND asus_segment NOT IN ('Consumer')"""

# Generate df from reseller_query
df_reseller_name_sql = pd.read_sql(reseller_name_query, engine)

# Filter df
reseller_list_df = reseller_list_df[~reseller_list_df['reseller_name'].isin(reseller_list_df['reseller_name'])]

# Remove duplicates on [reseller_name] to only upload unique values to reseller_list
reseller_list_df = reseller_list_df.drop_duplicates(subset=['reseller_name'], keep='first')

# Export to .csv
reseller_list_df.to_csv(r'../output/02_new_resellers.csv', index=False)

In [ ]:
#Insert only columsn existing in reseller_list
columns_to_insert = ['reseller_name', 'country_segment', 'asus_segment', 'country_id']

if not reseller_list_df.empty:
    reseller_list_df[columns_to_insert].to_sql(
        'reseller_list', engine, if_exists='append', index=False
    )
    print("Data inserted to 'reseller_list'.")
else:
    print("No data to upload. The DataFrame is empty.")

#### Insert new [reseller_code]

Step 1: Import from DB: reseller_code

In [ ]:
#----------------------------------------------------------------------------------------------
# Step 1 ) Filter out rows where [reseller_code] is already in our database
#----------------------------------------------------------------------------------------------

# SQL query from reseller_codes table to merge them into dataframe
reseller_code_query = f"""SELECT CAST(reseller_codes.id AS TEXT) AS reseller_code_id, importer_id,
                         reseller_code, reseller_origin, CAST(reseller_id AS TEXT), asus_segment
                        FROM reseller_codes
                        LEFT JOIN importer ON reseller_codes.importer_id = importer.id
                        LEFT JOIN reseller_list ON reseller_codes.reseller_id = reseller_list.id
                        WHERE importer.country_id = 1"""

df_reseller_code_sql = pd.read_sql(reseller_code_query, engine)
df_reseller_code_sql['reseller_origin'] = df_reseller_code_sql['reseller_origin'].astype('str').str.upper()

# Identify [reseller code] values that are not found in database
# For this we need to check the combination of [reseller_code, importer_id] using .merge
new_reseller_codes = reseller_formatted.merge(df_reseller_code_sql[['reseller_code', 'importer_id']],
                                              on=['reseller_code', 'importer_id'],how='left',indicator=True
)

# Drop indicator columns and duplicates
new_reseller_codes = new_reseller_codes[new_reseller_codes['_merge'] == 'left_only'].drop(columns=['_merge'])
new_reseller_codes = new_reseller_codes.drop_duplicates()
print(f'reseller_codes nuevos: {new_reseller_codes.shape[0]}')

In [ ]:
#----------------------------------------------------------------------------------------------
# Step 2 ) Get [reseller_id] from reseller_list
#----------------------------------------------------------------------------------------------

# SQL query to get reseller_name from database
reseller_name_query = f"""SELECT reseller_name, CAST(id AS TEXT) as reseller_id
                        FROM reseller_list
                        WHERE country_id = 1
                        AND asus_segment NOT IN ('Consumer')"""

# Generate df from reseller_query // Merge new_reseller_codes with df_reseller_code_sql to get [reseller_id]
df_reseller_name_sql = pd.read_sql(reseller_name_query, engine)
new_reseller_codes = pd.merge(new_reseller_codes, df_reseller_name_sql, on = 'reseller_name', how = 'left')

# Keep only columsn to be inserted and export to .csv
columns_to_insert = ['reseller_code', 'importer_id', 'reseller_origin', 'reseller_id', 'national_id']
reseller_codes_to_check = new_reseller_codes[columns_to_insert]
reseller_codes_to_check.to_csv(r'../output/02_reseller_codes_to_upload.csv', index=False)

#----------------------------------------------------------------------------------------------
# Step 3 ) Check for reseller codes that have being assigned to a reseller_id
#----------------------------------------------------------------------------------------------
reseller_codes_df_no_pcf = reseller_codes_to_check[reseller_codes_to_check['reseller_id'].notna() & (reseller_codes_to_check['reseller_id'] != '')]
reseller_codes_df_no_pcf.to_csv(r'../output/02_reseller_codes_to_upload_with_reseller_id.csv', index=False)

reseller_codes_to_insert_no_pcf = reseller_codes_df_no_pcf.copy()

#----------------------------------------------------------------------------------------------
# Step 4 ) Check for reseller codes that have not being assigned to a reseller_id that have same code but different reseller_name (PCF data)
#----------------------------------------------------------------------------------------------
reseller_codes_df_pcf = reseller_codes_to_check[~(reseller_codes_to_check['reseller_id'].notna() & (reseller_codes_to_check['reseller_id'] != ''))]
reseller_codes_df_pcf = reseller_codes_df_pcf.drop(columns = ['reseller_id'])

#----------------------------------------------------------------------------------------------
# Step 5 ) Check for reseller codes from PCF that have not being assigned to a reseller_id. Check for reseller_code scopes
#----------------------------------------------------------------------------------------------
reseller_codes_df_pcf = reseller_codes_df_pcf[reseller_codes_df_pcf['importer_id'] == 2]

# SQL query from reseller_codes table to merge them into dataframe
reseller_code_id_query = f"""SELECT reseller_code, CAST(reseller_id AS TEXT)
                        FROM reseller_codes
                        LEFT JOIN importer ON reseller_codes.importer_id = importer.id
                        LEFT JOIN reseller_list ON reseller_codes.reseller_id = reseller_list.id
                        WHERE importer.country_id = 1 and asus_segment NOT IN ('Consumer')"""

df_reseller_code_id_sql = pd.read_sql(reseller_code_id_query, engine)

df_reseller_code_id_sql_unique = df_reseller_code_id_sql[['reseller_code', 'reseller_id']].drop_duplicates(subset=['reseller_code'])
reseller_codes_to_insert_pcf= pd.merge(reseller_codes_df_pcf, df_reseller_code_id_sql_unique, on = 'reseller_code', how = 'left')
reseller_codes_to_insert_pcf.to_csv(r'../output/02_reseller_codes_to_upload_with_reseller_id_PCF.csv', index=False)

#----------------------------------------------------------------------------------------------
# Step 6 ) If reseller_id is Blank or NULL and importer_id = 2, then Change missing reseller_id into 9599 (PCF reseller) 
#----------------------------------------------------------------------------------------------
reseller_codes_to_insert_pcf.loc[(reseller_codes_to_insert_pcf['importer_id'] == 2) & 
                                 (reseller_codes_to_insert_pcf['reseller_id'].isna() | 
                                  (reseller_codes_to_insert_pcf['reseller_id'] == '') ),'reseller_id'] = 9599

#----------------------------------------------------------------------------------------------
# Step 7 ) Concat both dataframes to get final reseller_codes_to_insert
#----------------------------------------------------------------------------------------------
reseller_codes_to_insert = pd.concat([reseller_codes_to_insert_pcf, reseller_codes_to_insert_no_pcf], ignore_index= False)
reseller_codes_to_insert = reseller_codes_to_insert.drop_duplicates(subset=['reseller_code', 'importer_id'])
reseller_codes_to_insert.to_csv(r'../output/02_reseller_codes_to_insert.csv', index=False)

In [ ]:
#----------------------------------------------------------------------------------------------
# Step 3 ) Insert new reseller codes into reseller_codes table
#----------------------------------------------------------------------------------------------
if not reseller_codes_to_insert.empty:
    reseller_codes_to_insert.to_sql('reseller_codes', engine, if_exists='append', index=False)
    print("Data inserted to 'reseller_codes'.")
else:
    print("No data to upload. The DataFrame is empty.")

#### Upload new sales

Step 1: Import sales, stock, exrate, updated sku_disti and reseller_codes tables

In [ ]:
#SQL query for disti_sales
disti_sales_query = f"""SELECT date_sales, sku_id, invoice, reseller_code_id
                        FROM disti_sales
                        LEFT JOIN sku_disti ON disti_sales.sku_id = sku_disti.id
                        LEFT JOIN importer ON sku_disti.importer_id = importer.id
                        WHERE country_id = ({country_id_credential})"""

disti_sales_sql = pd.read_sql(disti_sales_query, engine)
disti_sales_sql['new_key'] =  (disti_sales_sql['date_sales'].astype(str)+'-'+
                               disti_sales_sql['sku_id'].astype(str)+'-'+
                               disti_sales_sql['invoice'].astype(str)+'-'+
                               disti_sales_sql['reseller_code_id'].astype(str))


# SQL query for ex_rate table with conversion from CLP to USD to merge them into dataframe
exrate_query = f"""SELECT date AS date_sales, currency_code, ex_rate
                   FROM world_exrates
                   WHERE currency_code = 'CLP' OR  currency_code = 'USD';
                   """
exrate_sql = pd.read_sql(exrate_query, engine)
exrate_sql['date_sales'] = pd.to_datetime(exrate_sql['date_sales'], errors='coerce')


# SQL query for sku_disti table with sku_id to merge them into dataframe
new_df_sku_disti_sql = pd.read_sql(sku_disti_query, engine)

# SQL query for reseller_codes table to merge them into dataframe. Use the same query from before.
new_df_reseller_sql = pd.read_sql(reseller_code_query, engine)

Step 2: SET sell_out_usd column to convert all sell_out_value into USD

In [ ]:
# Fix currency issues with PCF sales. Stock comes in USD
df_sales_currency = pd.merge(df_sales_sanitized, exrate_sql, on = ['date_sales', 'currency_code'], how = 'left')

df_sales_currency['sell_out_usd'] = df_sales_currency['sell_out_value'] / df_sales_currency['ex_rate']
df_sales_currency['sell_out_usd'] = round(df_sales_currency['sell_out_usd'], 2) #round to 2 decimals

Step 3: Format sales merging new sku_distis and new resellers

In [ ]:
# Filter df_sales_sanitized where so_units and so_value are equal to 0
df_sales_currency = df_sales_currency[~((df_sales_currency["sell_out_units"] == 0) & (df_sales_currency["sell_out_value"] == 0))]

#Merge sales with updated sku_distis and resellers tables. 
df_sales_merged = pd.merge(df_sales_currency, new_df_sku_disti_sql, on = ['sku', 'importer_id'] , how='left')
print('Rows after merging skus: ',df_sales_merged.shape[0])

df_sales_merged = pd.merge(df_sales_merged, new_df_reseller_sql, on = ['reseller_code', 'importer_id'], how='left')
print('Rows after merging reseller codes: ',df_sales_merged.shape[0])

#Group by 'date_sales', 'sku_id', 'invoice', 'reseller_code_id' sales to avoid duplicates
df_sales_grouped = df_sales_merged.groupby(['date_sales', 'sku_id', 'invoice', 'reseller_code_id', 'currency_code']).agg({'sell_out_units': 'sum', 'sell_out_value': 'sum', 'sell_out_usd': 'sum'}).reset_index()

df_sales_grouped.to_csv(r'../output/02_df_sales_grouped.csv', index=False)
print('Rows after grouping by date, sku and reseller_code: ', df_sales_grouped.shape[0])

df_sales_grouped['new_key'] = df_sales_grouped['date_sales'].astype(str)+'-'+df_sales_grouped['sku_id'].astype(str)+'-'+df_sales_grouped['invoice'].astype(str)+'-'+df_sales_grouped['reseller_code_id'].astype(str)

#Columns to be used for Sales: date_sales, sku_id, invoice, reseller_code_id, sell_out_units, sell_out_value
df_sales_to_upload = df_sales_grouped[['date_sales', 'sku_id', 'invoice', 'reseller_code_id', 'sell_out_units', 'sell_out_value', 'currency_code', 'sell_out_usd', 'new_key']]
df_sales_to_upload.to_csv(r'../output/02_df_sales_to_upload.csv', index=False)

Step 4: Check for sales with no reseller_code_id

In [ ]:
#Create mask to identify missing resellers for further analysis
columns_to_check_sales = ['reseller_code_id']

mask_sales_no_reseller = (df_sales_to_upload[columns_to_check_sales] == "").any(axis=1)
sales_to_check = df_sales_to_upload[mask_sales_no_reseller]
print(sales_to_check.shape)

sales_to_check.to_csv(r'../output/02_sales_missing_code_id.csv', index=False)

#Define rows to be uploaded in sales
df_sales_to_upload = df_sales_to_upload[~mask_sales_no_reseller]
print(df_sales_to_upload.shape)
df_sales_to_upload.to_csv(r'../output/02_final_sales_to_upload.csv', index=False)

Step 5: create merge to look up for changes in sales. Divide them into different DFs

In [ ]:
# Merging both dataframes to obtain old sales
new_sales_to_upload = df_sales_to_upload[~df_sales_to_upload['new_key'].isin(disti_sales_sql['new_key'])]
print(new_sales_to_upload.shape)
new_sales_to_upload.to_csv(r'../output/02_df_new_sales_to_upload.csv', index=False)
new_sales_to_upload = new_sales_to_upload[['date_sales', 'sku_id', 'invoice', 'reseller_code_id', 'sell_out_units', 'sell_out_value', 'currency_code', 'sell_out_usd']]

old_sales_to_update = df_sales_to_upload[df_sales_to_upload['new_key'].isin(disti_sales_sql['new_key'])]
print(old_sales_to_update.shape)

old_sales_to_update.to_csv(r'../output/02_df_old_sales_to_update.csv', index=False)
old_sales_to_update = old_sales_to_update[['date_sales', 'sku_id', 'invoice', 'reseller_code_id', 'sell_out_units', 'sell_out_value', 'currency_code', 'sell_out_usd']]

Step 6: Upload new sales into database table

In [ ]:
if not new_sales_to_upload.empty:
    new_sales_to_upload.to_sql('disti_sales', engine, if_exists='append', index=False)
    print("Data successfully inserted.")
else:
    print("No data to inserted. The DataFrame is empty.")

Step 7: Update existing sales by sell_out_units, sell_out_value for date_sales, sku_id and invoice

In [ ]:
update_query = """
UPDATE disti_sales
SET sell_out_units = :sell_out_units,
sell_out_value = :sell_out_value,
currency_code = :currency_code,
sell_out_usd = :sell_out_usd
WHERE date_sales = :date_sales 
AND sku_id = :sku_id 
AND invoice = :invoice
AND reseller_code_id = :reseller_code_id;
"""
with engine.connect() as connection:
    for _, row in old_sales_to_update.iterrows():
        connection.execute(
            text(update_query),
            {
                'date_sales': row['date_sales'],
                'sku_id': row['sku_id'],
                'invoice': row['invoice'],
                'reseller_code_id': row['reseller_code_id'],
                'sell_out_units': row['sell_out_units'],
                'sell_out_value': row['sell_out_value'],
                'currency_code': row['currency_code'],
                'sell_out_usd': row['sell_out_usd']
            }
        )
        connection.commit()

#### Upload new stock

Step 1: Import from DB: stock_disti

In [ ]:
#SQL query for disti_stock
stock_disti_query = f"""SELECT date_stock, sku_id, stock_units, stock_value 
                    FROM disti_stock 
                    LEFT JOIN sku_disti on sku_id = sku_disti.id 
                    LEFT JOIN importer on sku_disti.importer_id = importer.id 
                    WHERE country_id = 1"""

stock_disti_sql = pd.read_sql(stock_disti_query, engine)

#SQL query for sku and importer 
sku_importer_new_query = f"""SELECT sku_disti.id AS sku_id, sku, importer_id 
                            FROM sku_disti
                            LEFT JOIN importer ON sku_disti.importer_id = importer.id
                            WHERE country_id = 1"""

new_df_sku_importer_sql = pd.read_sql(sku_importer_new_query, engine)

#Left join ALL stock to obtain importer from sku_id
stock_disti_merged = pd.merge(stock_disti_sql, new_df_sku_importer_sql, on = 'sku_id', how='left')

stock_disti_merged['sku_id'] = stock_disti_merged['sku_id'].astype(int)
stock_disti_merged['importer_id'] = stock_disti_merged['importer_id'].astype(int) 

#Create parameter "new_key" for comparison
stock_disti_merged['new_key'] =  (stock_disti_merged['date_stock'].astype(str) + '-' + 
                                  stock_disti_merged['sku_id'].astype(str) +
                                 '-' + stock_disti_merged['importer_id'].astype(str))

Step 2: SET stock_usd from stock value and applying ex_rate

In [ ]:
# Fix currency issues with PCF sales. Stock comes in USD
exrate_sql = exrate_sql.rename(columns={'date_sales': 'date_stock'})
df_stock_currency = pd.merge(df_stock_sanitized, exrate_sql, on = ['date_stock', 'currency_code'], how = 'left')

df_stock_currency['stock_usd'] = df_stock_currency['stock_value'] * df_stock_currency['ex_rate']
df_stock_currency['stock_usd'] = round(df_stock_currency['stock_usd'], 2) #round to 2 decimals

Step 3: Format stock merging new sku_distis

In [ ]:
# Filter df_stock_sanitized where units and value are equal to 0
df_stock_sanitized = df_stock_currency[~((df_stock_currency["stock_units"] == 0) & (df_stock_currency["stock_value"] == 0))]

# Merge stock with updated sku_distis table.
df_stock_merged = pd.merge(df_stock_sanitized, new_df_sku_disti_sql, on = ['sku', 'importer_id'], how='left')
df_stock_merged['new_key'] = (df_stock_merged['date_stock'].astype(str)+ '-' +
                              df_stock_merged['sku_id'].astype(str) + '-' +
                              df_stock_merged['importer_id'].astype(str))  

#Columns to be used for Sales: date_stock, sku_id, stock_units, stock_value
df_stock_to_upload = df_stock_merged[['date_stock', 'sku_id', 'stock_units', 'stock_value', 'new_key', 'currency_code', 'stock_usd']]
df_stock_to_upload.to_csv(r'../output/02_df_stock_to_upload.csv', index=False)

Step 4: create merge to look up for changes in stock. Divide them into different DFs

In [ ]:
# Merging both dataframes to obtain old stock
new_stock_to_upload = df_stock_to_upload[~df_stock_to_upload['new_key'].isin(stock_disti_merged['new_key'])]
print(new_stock_to_upload.shape)
new_stock_to_upload.to_csv(r'../output/02_df_new_stock_to_upload.csv', index=False)
new_stock_to_upload = new_stock_to_upload[['date_stock', 'sku_id', 'stock_units', 'stock_value', 'currency_code', 'stock_usd']]

old_stock_to_update = df_stock_to_upload[df_stock_to_upload['new_key'].isin(stock_disti_merged['new_key'])]
print(old_stock_to_update.shape)

old_stock_to_update.to_csv(r'../output/02_df_old_stock_to_update.csv', index=False)
old_stock_to_update = old_stock_to_update[['date_stock', 'sku_id', 'stock_units', 'stock_value', 'currency_code', 'stock_usd']]

Step 5: Upload new stock into stock distributor table

In [ ]:
if not new_stock_to_upload.empty:
    new_stock_to_upload.to_sql('disti_stock', engine, if_exists='append', index=False)
    print("Data inserted to 'disti_stock'.")
else:
    print("No data to upload. The DataFrame is empty.")

Step 6: Update existing stock by stock_units, stock_value for date_stock and sku_id

In [ ]:
update_query = """
UPDATE disti_stock
SET stock_units = :stock_units,
stock_value = :stock_value,
currency_code = :currency_code,
stock_usd = :stock_usd
WHERE date_stock = :date_stock 
AND sku_id = :sku_id;
"""
with engine.connect() as connection:
    for _, row in old_stock_to_update.iterrows():
        connection.execute(
            text(update_query),
            {
                'date_stock': row['date_stock'],
                'sku_id': row['sku_id'],
                'stock_units': row['stock_units'],
                'stock_value': row['stock_value'],
                'currency_code': row['currency_code'],
                'stock_usd': row['stock_usd']
            }
        )
        connection.commit()

In [ ]:
#Close connection to asus_db
engine.dispose()